In [1]:
import os
import sys
import requests
import pandas as pd
from dotenv import load_dotenv

sys.path.append("../src")
import sec_extract

load_dotenv()
HEADERS = {"User-Agent": os.getenv("SEC_USER_AGENT")}

url = "https://data.sec.gov/api/xbrl/companyfacts/CIK0001436126.json"
data = requests.get(url, headers=HEADERS, timeout=30).json()

mistras = sec_extract.extract_company(data, "1231")
print("Rows extracted:", len(mistras))
mistras[mistras["fiscal_year"] == 2025]

Rows extracted: 158


,fiscal_year,end,value_000s,tag,accn,variable
9,2025,2025-12-31,724024.0,RevenueFromContractWithCustomerExcludingAssess...,0001628280-26-016765,revenue
19,2025,2025-12-31,40572.0,OperatingIncomeLoss,0001628280-26-016765,operating_income
28,2025,2025-12-31,14643.0,InterestExpenseNonoperating,0001628280-26-016765,interest_expense
38,2025,2025-12-31,32058.0,WeightedAverageNumberOfDilutedSharesOutstanding,0001628280-26-016765,diluted_shares
48,2025,2025-12-31,31008.0,DepreciationDepletionAndAmortization,0001628280-26-016765,d_and_a
57,2025,2025-12-31,32981.0,NetCashProvidedByUsedInOperatingActivities,0001628280-26-016765,cfo
66,2025,2025-12-31,24674.0,PaymentsToAcquirePropertyPlantAndEquipment,0001628280-26-016765,capex
83,2025,2025-12-31,28008.0,CashAndCashEquivalentsAtCarryingValue,0001628280-26-016765,cash
93,2025,2025-12-31,12849.0,LongTermDebtCurrent,0001628280-26-016765,short_term_debt
103,2025,2025-12-31,165143.0,LongTermDebtNoncurrent,0001628280-26-016765,long_term_debt


In [4]:
answer_key_2025 = {
    "revenue": 724024, "operating_income": 40572, "interest_expense": 14643,
    "diluted_shares": 32058, "d_and_a": 31008, "cfo": 32981, "capex": 24674,
    "acquisitions_paid": 0, "cash": 28008, "short_term_debt": 12849,
    "long_term_debt": 165143, "finance_lease_current": 7025,
    "finance_lease_noncurrent": 17340, "stockholders_equity": 235104,
    "accounts_receivable": 154673, "goodwill": 184829,
}

fy2025 = mistras[mistras["fiscal_year"] == 2025].set_index("variable")["value_000s"]

for variable, expected in answer_key_2025.items():
    actual = fy2025.get(variable)
    if actual is None:
        status = "MISSING"
    elif abs(actual - expected) < 1:
        status = "MATCH"
    else:
        status = "MISMATCH"
    print(f"{variable:26} key: {expected:>8}   api: {actual}   {status}")

revenue                    key:   724024   api: 724024.0   MATCH
operating_income           key:    40572   api: 40572.0   MATCH
interest_expense           key:    14643   api: 14643.0   MATCH
diluted_shares             key:    32058   api: 32058.0   MATCH
d_and_a                    key:    31008   api: 31008.0   MATCH
cfo                        key:    32981   api: 32981.0   MATCH
capex                      key:    24674   api: 24674.0   MATCH
acquisitions_paid          key:        0   api: None   MISSING
cash                       key:    28008   api: 28008.0   MATCH
short_term_debt            key:    12849   api: 12849.0   MATCH
long_term_debt             key:   165143   api: 165143.0   MATCH
finance_lease_current      key:     7025   api: 7025.0   MATCH
finance_lease_noncurrent   key:    17340   api: 17340.0   MATCH
stockholders_equity        key:   235104   api: 235104.0   MATCH
accounts_receivable        key:   154673   api: 154673.0   MATCH
goodwill                   key:   1848

In [5]:
import json
import time

CACHE_DIR = "../data/raw/companyfacts"
os.makedirs(CACHE_DIR, exist_ok=True)

universe = pd.read_csv("../data/processed/universe_sic_filtered.csv")
print("Companies in universe:", len(universe))


def load_company_facts(cik):
    """Load a company's facts from the local cache, downloading only if not saved yet."""
    path = f"{CACHE_DIR}/CIK{cik:010d}.json"
    if os.path.exists(path):
        with open(path, encoding="utf-8") as f:
            return json.load(f)

    url = f"https://data.sec.gov/api/xbrl/companyfacts/CIK{cik:010d}.json"
    response = requests.get(url, headers=HEADERS, timeout=60)
    time.sleep(0.12)   # stay under the SEC's rate limit
    if response.status_code != 200:
        return None    # e.g. a company with no XBRL financial data

    with open(path, "w", encoding="utf-8") as f:
        f.write(response.text)
    return response.json()

Companies in universe: 182


In [6]:
all_tables = []
log = []

for i, company in enumerate(universe.itertuples(), start=1):
    cik, ticker = company.cik, company.ticker
    try:
        data = load_company_facts(cik)
        if data is None:
            log.append([cik, ticker, "NO COMPANY FACTS"])
            continue

        # Fiscal year-end: from the SEC profile, or inferred from the 10-K filings
        if pd.isna(company.fiscal_year_end):
            fye = sec_extract.infer_fiscal_year_end(data)
            if fye is None:
                log.append([cik, ticker, "NO FISCAL YEAR END"])
                continue
        else:
            fye = f"{int(company.fiscal_year_end):04d}"   # e.g. 630 becomes "0630"

        table = sec_extract.extract_company(data, fye)
        if table is None:
            log.append([cik, ticker, "NO ANNUAL 10-K DATA"])
            continue

        table.insert(0, "cik", cik)
        table.insert(1, "ticker", ticker)
        all_tables.append(table)
        log.append([cik, ticker, "OK"])

    except Exception as error:
        log.append([cik, ticker, f"ERROR: {error}"])

    if i % 25 == 0 or i == len(universe):
        print(f"Processed {i} of {len(universe)}")

Processed 50 of 182
Processed 75 of 182
Processed 100 of 182
Processed 150 of 182
Processed 182 of 182


In [7]:
financials = pd.concat(all_tables, ignore_index=True)
financials.to_csv("../data/processed/financials_long.csv", index=False)

status = pd.DataFrame(log, columns=["cik", "ticker", "status"])
status.to_csv("../data/processed/extraction_log.csv", index=False)

print("Rows saved:", len(financials))
print("Companies with data:", financials["cik"].nunique())
status["status"].value_counts()

Rows saved: 18684
Companies with data: 125


status
OK                     125
NO ANNUAL 10-K DATA     44
NO FISCAL YEAR END      13
Name: count, dtype: int64

In [8]:
   financials[(financials["ticker"] == "MG") & (financials["fiscal_year"] == 2025) & (financials["variable"] == "revenue")]

,cik,ticker,fiscal_year,end,value_000s,tag,accn,variable
12651,1436126,MG,2025,2025-12-31,724024.0,RevenueFromContractWithCustomerExcludingAssess...,0001628280-26-016765,revenue


In [9]:
US_STATES = {"AL","AK","AZ","AR","CA","CO","CT","DE","DC","FL","GA","HI","ID","IL","IN","IA","KS",
             "KY","LA","ME","MD","MA","MI","MN","MS","MO","MT","NE","NV","NH","NJ","NM","NY","NC",
             "ND","OH","OK","OR","PA","RI","SC","SD","TN","TX","UT","VT","VA","WA","WV","WI","WY"}


def annual_forms_filed(cik):
    """List the annual-report forms (10-K, 20-F, 40-F) found in a company's cached facts."""
    path = f"{CACHE_DIR}/CIK{cik:010d}.json"
    if not os.path.exists(path):
        return "no file"
    with open(path, encoding="utf-8") as f:
        data = json.load(f)
    forms = set()
    for tags in data.get("facts", {}).values():
        for info in tags.values():
            for entries in info["units"].values():
                forms.update(e.get("form") for e in entries)
    annual = sorted(f for f in forms if f in {"10-K", "10-K/A", "20-F", "40-F", "10-KT"})
    return ", ".join(annual) if annual else "none"


problems = status[status["status"] != "OK"].merge(
    universe[["cik", "company_name", "sic_code", "hq_state", "fiscal_year_end"]], on="cik")
problems["us_hq"] = problems["hq_state"].isin(US_STATES)
problems["annual_forms"] = problems["cik"].apply(annual_forms_filed)

pd.set_option("display.max_rows", 100)
print(problems.groupby(["status", "us_hq"]).size())
problems[["ticker", "company_name", "status", "hq_state", "us_hq", "annual_forms"]]

status               us_hq
NO ANNUAL 10-K DATA  False    42
                     True      2
NO FISCAL YEAR END   False    13
dtype: int64


,ticker,company_name,status,hq_state,us_hq,annual_forms
0,BABA,Alibaba Group Holding Ltd,NO FISCAL YEAR END,NaN,False,20-F
1,PDD,PDD Holdings Inc.,NO ANNUAL 10-K DATA,NaN,False,20-F
2,RELX,RELX PLC,NO ANNUAL 10-K DATA,X0,False,20-F
3,SE,Sea Ltd,NO ANNUAL 10-K DATA,U0,False,20-F
4,TCOM,Trip.com Group Ltd,NO ANNUAL 10-K DATA,U0,False,20-F
5,KSPI,Joint Stock Co Kaspi.kz,NO ANNUAL 10-K DATA,1P,False,20-F
6,GIB,CGI INC,NO ANNUAL 10-K DATA,A8,False,40-F
7,GRAB,Grab Holdings Ltd,NO ANNUAL 10-K DATA,U0,False,20-F
8,RTO,RENTOKIL INITIAL PLC /FI,NO ANNUAL 10-K DATA,NaN,False,20-F
9,WSE,Wise Group plc,NO ANNUAL 10-K DATA,NaN,False,20-F


In [8]:
   status[status["ticker"] == "BWMN"]

,cik,ticker,status
89,1847590,BWMN,OK


In [10]:
def exclusion_reason(row):
    """Explain why a company does not continue in the screen (blank if it does)."""
    if row["status"] == "OK":
        return ""
    if "20-F" in row["annual_forms"] or "40-F" in row["annual_forms"]:
        return "Foreign private issuer (files 20-F/40-F); mandate requires U.S. companies"
    if not row["us_hq"]:
        return "Non-U.S. headquarters; mandate requires U.S. companies"
    return "No annual 10-K financial data (e.g., recent listing)"


review = universe.merge(status[["cik", "status"]], on="cik")
review["us_hq"] = review["hq_state"].isin(US_STATES)
review["annual_forms"] = review["cik"].apply(annual_forms_filed)
review["exclusion_reason"] = review.apply(exclusion_reason, axis=1)
review["in_universe"] = review["status"] == "OK"

review.to_csv("../data/processed/universe_review.csv", index=False)

print(review["in_universe"].value_counts())
review["exclusion_reason"].value_counts()

in_universe
True     125
False     57
Name: count, dtype: int64


exclusion_reason
                                                                             125
Foreign private issuer (files 20-F/40-F); mandate requires U.S. companies     55
No annual 10-K financial data (e.g., recent listing)                           1
Non-U.S. headquarters; mandate requires U.S. companies                         1
Name: count, dtype: int64

In [11]:
review[review["exclusion_reason"].str.startswith("Non-U.S.")][["ticker", "company_name", "hq_state", "annual_forms"]]

,ticker,company_name,hq_state,annual_forms
129,BBCI,Bradbury Capital Inc.,NaN,none
